# Silver Layer - Data Transformation

This notebook transforms bronze tables into silver tables with data quality improvements and derived columns.

In [0]:
from pyspark.sql.functions import col, when

## Constants

In [0]:
CATALOG_NAME = 'workspace'
SCHEMA_NAME__BRONZE = 'olist_bronze'
SCHEMA_NAME__SILVER = 'olist_silver'

# Create Silver Schema

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")

qualified_silver_schema_name = f"{CATALOG_NAME}.{SCHEMA_NAME__SILVER}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {qualified_silver_schema_name}")

# Create Master Tables

## silver.Status (Pass Through)

In [0]:
# Pass through from bronze - no transformations needed
table__bronze__status = f"{CATALOG_NAME}.{SCHEMA_NAME__BRONZE}.status"
table__silver__status = f"{CATALOG_NAME}.{SCHEMA_NAME__SILVER}.status"

status_df = spark.table(table__bronze__status)

status_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable(table__silver__status)

## silver.Products (Add Active Flag)

In [0]:
# Transform products: remove deleted_at, add active boolean column
table__bronze__products = f"{CATALOG_NAME}.{SCHEMA_NAME__BRONZE}.products"
table__silver__products = f"{CATALOG_NAME}.{SCHEMA_NAME__SILVER}.products"

products_bronze_df = spark.table(table__bronze__products)

# Add active column and drop deleted_at
products_silver_df = products_bronze_df \
  .withColumn("active", when(col("deleted_at").isNull(), True).otherwise(False)) \
  .drop("deleted_at")

products_silver_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable(table__silver__products)

## silver.Customers (Add Active Flag)

In [0]:
# Transform customers: remove deleted_at, add active boolean column
table__bronze__customers = f"{CATALOG_NAME}.{SCHEMA_NAME__BRONZE}.customers"
table__silver__customers = f"{CATALOG_NAME}.{SCHEMA_NAME__SILVER}.customers"

customers_bronze_df = spark.table(table__bronze__customers)

# Add active column and drop deleted_at
customers_silver_df = customers_bronze_df \
  .withColumn("active", when(col("deleted_at").isNull(), True).otherwise(False)) \
  .drop("deleted_at")

customers_silver_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable(table__silver__customers)

# Verify Silver Tables

In [0]:
%sql
SELECT type, COUNT(*) AS total_status 
FROM workspace.olist_silver.status 
GROUP BY type;

In [0]:
%sql
SELECT 
  active,
  COUNT(*) AS product_count
FROM workspace.olist_silver.products
GROUP BY active;

In [0]:
%sql
SELECT 
  active,
  COUNT(*) AS customer_count
FROM workspace.olist_silver.customers
GROUP BY active;

# Create Transaction Tables

## silver.Orders (Split by Status + UTC Conversion)

In [0]:
from pyspark.sql.functions import to_utc_timestamp

# Transform orders: convert datetime to UTC and split by status
table__bronze__orders = f"{CATALOG_NAME}.{SCHEMA_NAME__BRONZE}.orders"
table__silver__orders_active = f"{CATALOG_NAME}.{SCHEMA_NAME__SILVER}.orders__active"
table__silver__orders_inactive = f"{CATALOG_NAME}.{SCHEMA_NAME__SILVER}.orders__inactive"

orders_bronze_df = spark.table(table__bronze__orders)

# Get datetime columns to convert to UTC
datetime_columns = [field.name for field in orders_bronze_df.schema.fields 
                   if str(field.dataType) == 'TimestampType']

print(f"Converting datetime columns to UTC: {datetime_columns}")

# Convert all datetime columns to UTC
orders_df = orders_bronze_df
for col_name in datetime_columns:
    orders_df = orders_df.withColumn(col_name, to_utc_timestamp(col(col_name), 'UTC'))

# Split into active and inactive tables
orders_active_df = orders_df.filter(col("order_status") != "Cancelled")
orders_inactive_df = orders_df.filter(col("order_status") == "Cancelled")

# Write active orders
orders_active_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable(table__silver__orders_active)

# Write inactive orders
orders_inactive_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable(table__silver__orders_inactive)

print(f"✓ Created {table__silver__orders_active}: {orders_active_df.count()} rows")
print(f"✓ Created {table__silver__orders_inactive}: {orders_inactive_df.count()} rows")

## silver.Order_Items (Add Discounted Flag + UTC Conversion)

In [0]:
# Transform order_items: convert datetime to UTC and add discounted boolean
table__bronze__order_items = f"{CATALOG_NAME}.{SCHEMA_NAME__BRONZE}.order_items"
table__silver__order_items = f"{CATALOG_NAME}.{SCHEMA_NAME__SILVER}.order_items"

order_items_bronze_df = spark.table(table__bronze__order_items)

# Get datetime columns to convert to UTC
datetime_columns = [field.name for field in order_items_bronze_df.schema.fields 
                   if str(field.dataType) == 'TimestampType']

print(f"Converting datetime columns to UTC: {datetime_columns}")

# Convert all datetime columns to UTC
order_items_df = order_items_bronze_df
for col_name in datetime_columns:
    order_items_df = order_items_df.withColumn(col_name, to_utc_timestamp(col(col_name), 'UTC'))

# Add discounted column and drop discount_applied
order_items_silver_df = order_items_df \
  .withColumn("discounted", when(col("discount_applied") > 0, True).otherwise(False)) \
  .drop("discount_applied")

order_items_silver_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable(table__silver__order_items)

print(f"✓ Created {table__silver__order_items}: {order_items_silver_df.count()} rows")

# Verify Transaction Tables

In [0]:
%sql
SELECT 
  order_status,
  COUNT(*) AS order_count
FROM workspace.olist_silver.orders__active
GROUP BY order_status;

In [0]:
%sql
SELECT 
  order_status,
  COUNT(*) AS order_count
FROM workspace.olist_silver.orders__inactive
GROUP BY order_status;

In [0]:
%sql
SELECT 
  discounted,
  COUNT(*) AS item_count
FROM workspace.olist_silver.order_items
GROUP BY discounted;